# animations — Multi-view → 3D on your Colab GPU (experimental)

Generate a **truer 3D shape** from up to four photos of your device
(front / back / left / right) using **Hunyuan3D-2mv** — a much stronger model
than TripoSR, conditioned on multiple views.

**Steps:** Runtime → Change runtime type → **GPU (T4 or better)**, then
Runtime → **Run all**. When the last cell prints a `gradio.live` URL, paste it
into the Studio's **Connect a generator** field.

Honest notes:
- First run installs + downloads several GB → **~10 minutes** before the URL appears.
- v1 makes **untextured geometry** (solid color) — much more accurate shape;
  the texture stage is a later upgrade. For a quick textured (rougher) model,
  use the single-view TripoSR notebook instead.
- Experimental: if a cell errors, copy the red text — dependency tweaks are
  normal on Colab and quick to patch.
- The link dies when this notebook stops; every launch prints a new one.


In [ ]:
# 1) Install Hunyuan3D-2 + a Studio-compatible server stack (~5-8 min)
%cd /content
!rm -rf Hunyuan3D-2
!git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q -e .

# Same known-good pins as the TripoSR notebook (gradio 4.44 dialect the Studio
# speaks, pydantic that doesn't break /info, numpy that Colab's chain accepts):
!pip install -q -U "gradio==4.44.1" "gradio-client==1.3.0" "huggingface_hub==0.25.2" \
    "pydantic==2.10.6" "trimesh>=4.5" "numpy==2.0.2" onnxruntime


In [ ]:
# 2) Multi-view app the Studio drives (api_name="generate_mv"), with share link.
%cd /content/Hunyuan3D-2
import os, tempfile
import gradio as gr

from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

print("Loading Hunyuan3D-2mv (first time downloads several GB)...")
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained("tencent/Hunyuan3D-2mv")
remover = BackgroundRemover()

def generate_mv(front, back, left, right):
    views = {}
    for name, img in (("front", front), ("back", back), ("left", left), ("right", right)):
        if img is None:
            continue
        img = img.convert("RGBA")
        # Remove the background unless the photo already has transparency.
        if img.getextrema()[3][0] == 255:
            img = remover(img.convert("RGB"))
        views[name] = img
    if "front" not in views:
        raise gr.Error("A front view is required.")
    mesh = pipe(image=views)[0]
    path = os.path.join(tempfile.mkdtemp(), "model.glb")
    mesh.export(path)
    return path

with gr.Blocks(title="animations - multi-view to 3D") as demo:
    gr.Markdown("## animations - multi-view to 3D (Hunyuan3D-2mv)\nDriven by the Studio - you can also test here directly.")
    with gr.Row():
        f = gr.Image(label="Front *", type="pil", image_mode="RGBA")
        b = gr.Image(label="Back", type="pil", image_mode="RGBA")
        l = gr.Image(label="Left", type="pil", image_mode="RGBA")
        r = gr.Image(label="Right", type="pil", image_mode="RGBA")
    out = gr.Model3D(label="Result (.glb)")
    gr.Button("Generate", variant="primary").click(generate_mv, [f, b, l, r], out, api_name="generate_mv")

print("Watch for:  Running on public URL: https://XXXX.gradio.live")
demo.queue(max_size=4).launch(share=True)


## Use it
1. Copy the `https://XXXX.gradio.live` URL printed above.
2. Open the Studio: **zurlazar.github.io/animations/studio.html**
3. Paste the URL into **Connect a generator** -> **Save**.
4. Fill the **Front** slot (required) and as many of Back/Left/Right as you have
   -> **Generate 3D model**.

### Photo tips (these matter more than anything else)
- Put the device **alone** on a plain background - no cables, no clamps, no
  second instrument attached.
- Hold the phone **upright** (a sideways photo becomes a sideways model).
- Even, soft lighting; avoid deep shadows on a dark device.
- Shoot all views at roughly the **same distance and height**.
